<a href="https://colab.research.google.com/github/detektor777/colab_list_video/blob/main/supir__video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
!rm -rf SUPIR
# Clear cache from previous attempts
!rm -rf ~/.cache/huggingface/hub/models--laion* ~/.cache/clip ~/.cache/torch/hub/checkpoints

# Clone the official SUPIR repository
!git clone https://github.com/Fanghua-Yu/SUPIR.git
%cd SUPIR

# Install needed packages for SUPIR + Video tools from Real-ESRGAN
!pip install -q pytorch-lightning omegaconf einops timm open-clip-torch==2.24.0 k-diffusion bitsandbytes transformers accelerate diffusers scikit-image huggingface_hub ffmpeg-python imageio ipywidgets

import os
import re
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs("checkpoints", exist_ok=True)

# DOWNLOAD THE PHOTOREALISTIC BASE JUGGERNAUT-XL V9
print("Downloading the photorealistic base Juggernaut-XL v9 (~6.6 GB)...")
sdxl_path = hf_hub_download(
    repo_id="RunDiffusion/Juggernaut-XL-v9",
    filename="Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
    local_dir="checkpoints"
)

# DOWNLOAD SUPIR-v0Q WEIGHTS
print("Downloading SUPIR-v0Q weights (~5.3 GB)...")
supir_q_path = hf_hub_download(
    repo_id="camenduru/SUPIR",
    filename="SUPIR-v0Q.ckpt",
    local_dir="checkpoints"
)

# DOWNLOAD SUPIR-v0F WEIGHTS
print("Downloading SUPIR-v0F weights (~5.3 GB)...")
supir_f_path = hf_hub_download(
    repo_id="camenduru/SUPIR",
    filename="SUPIR-v0F.ckpt",
    local_dir="checkpoints"
)

print("Downloading LLaVA-v1.5-7B (~13 GB, once, then cached)...")
llava_path = snapshot_download(
    repo_id="liuhaotian/llava-v1.5-7b",
    local_dir="/root/llava-v1.5-7b",
)

print("Weights downloaded:", sdxl_path, supir_q_path, supir_f_path, llava_path)

# Patch 1: CKPT_PTH.py
ckpt_pth_content = f'''
LLAVA_CLIP_PATH = None
LLAVA_MODEL_PATH = "{llava_path}"
SDXL_CLIP1_PATH = None
SDXL_CLIP2_CKPT_PTH = None
'''
with open("CKPT_PTH.py", "w") as f:
    f.write(ckpt_pth_content)

# Patch 2: options/SUPIR_v0.yaml
with open("options/SUPIR_v0.yaml", "r") as f:
    cfg = f.read()
cfg = cfg.replace("\r\n", "\n")
cfg = re.sub(r"SDXL_CKPT:.*", f"SDXL_CKPT: {sdxl_path}", cfg)
cfg = re.sub(r"SUPIR_CKPT_Q:.*", f"SUPIR_CKPT_Q: {supir_q_path}", cfg)
cfg = re.sub(r"SUPIR_CKPT_F:.*", f"SUPIR_CKPT_F: {supir_f_path}", cfg)
with open("options/SUPIR_v0.yaml", "w") as f:
    f.write(cfg)

# Patch 2b: llava/model/language_model/llava_llama.py
llava_llama_path = os.path.join("llava", "model", "language_model", "llava_llama.py")
if os.path.exists(llava_llama_path):
    with open(llava_llama_path, "r") as f:
        llava_llama_content = f.read()
    llava_llama_content = llava_llama_content.replace("\r\n", "\n")

    old_register = 'AutoConfig.register("llava", LlavaConfig)'
    new_register = 'AutoConfig.register("llava", LlavaConfig, exist_ok=True)'
    if old_register in llava_llama_content:
        llava_llama_content = llava_llama_content.replace(old_register, new_register)
    with open(llava_llama_path, "w") as f:
        f.write(llava_llama_content)

# Patch 2c: llava/model/__init__.py
llava_init_path = os.path.join("llava", "model", "__init__.py")
with open(llava_init_path, "r") as f:
    llava_init_content = f.read()
lines = llava_init_content.splitlines()
new_lines = [ln for ln in lines if "mpt" not in ln.lower()]
llava_init_content_new = "\n".join(new_lines) + "\n"
with open(llava_init_path, "w") as f:
    f.write(llava_init_content_new)

# Patch 2d: llava/model/builder.py
builder_path = os.path.join("llava", "model", "builder.py")
with open(builder_path, "r") as f:
    builder_content = f.read()
builder_content = builder_content.replace("\r\n", "\n")

old_8bit_block = "    if load_8bit:\n        kwargs['load_in_8bit'] = True"
new_8bit_block = (
    "    if load_8bit:\n"
    "        from transformers import BitsAndBytesConfig\n"
    "        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)"
)
if old_8bit_block in builder_content:
    builder_content = builder_content.replace(old_8bit_block, new_8bit_block)
else:
    builder_content = builder_content.replace(
        "kwargs['load_in_8bit'] = True",
        "kwargs['quantization_config'] = __import__('transformers').BitsAndBytesConfig(load_in_8bit=True)"
    )

old_4bit_line = "kwargs['load_in_4bit'] = True"
if old_4bit_line in builder_content:
    builder_content = builder_content.replace(
        old_4bit_line,
        "kwargs['quantization_config'] = __import__('transformers').BitsAndBytesConfig(load_in_4bit=True)"
    )
with open(builder_path, "w") as f:
    f.write(builder_content)

# Patch 2e: llava/model/llava_arch.py
llava_arch_path = os.path.join("llava", "model", "llava_arch.py")
with open(llava_arch_path, "r") as f:
    llava_arch_content = f.read()
llava_arch_content = llava_arch_content.replace("\r\n", "\n")
old_expr = "past_key_values[-1][-1].shape[-2] + 1"
new_expr = (
    "(past_key_values.get_seq_length() if hasattr(past_key_values, 'get_seq_length') "
    "else past_key_values[-1][-1].shape[-2]) + 1"
)
if old_expr in llava_arch_content:
    llava_arch_content = llava_arch_content.replace(old_expr, new_expr)
with open(llava_arch_path, "w") as f:
    f.write(llava_arch_content)

# Patch 3: test.py
with open("test.py", "r") as f:
    content = f.read()
content = content.replace("\r\n", "\n")
content = content.replace(
    "from llava.llava_agent import LLavaAgent",
    "try:\n    from llava.llava_agent import LLavaAgent\nexcept Exception as e:\n    import traceback\n    print('LLaVA skipped (import error):')\n    traceback.print_exc()\n    LLavaAgent = None"
)
old_line = "    llava_agent = LLavaAgent(LLAVA_MODEL_PATH, device=LLaVA_device, load_8bit=args.load_8bit_llava, load_4bit=False)"
new_line = (
    "    if LLavaAgent is not None:\n"
    "        try:\n"
    "            llava_agent = LLavaAgent(LLAVA_MODEL_PATH, device=LLaVA_device, load_8bit=args.load_8bit_llava, load_4bit=False)\n"
    "        except Exception as e:\n"
    "            import traceback\n"
    "            print('LLaVA skipped (agent creation error):')\n"
    "            traceback.print_exc()\n"
    "            llava_agent = None\n"
    "    else:\n"
    "        llava_agent = None"
)
if old_line in content:
    content = content.replace(old_line, new_line)

# Patch 4: test.py - VRAM cleanup
old_captions_block = (
    "        if use_llava:\n"
    "            captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
    "        else:\n"
    "            captions = ['']"
)
new_captions_block = (
    "        if use_llava and llava_agent is not None:\n"
    "            try:\n"
    "                captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
    "                print(f'>>> LLaVA CAPTION: {captions}')\n"
    "                if hasattr(llava_agent, \"model\"):\n"
    "                    del llava_agent.model\n"
    "                del llava_agent\n"
    "                llava_agent = None\n"
    "                import torch, gc\n"
    "                gc.collect()\n"
    "                torch.cuda.empty_cache()\n"
    "            except Exception as e:\n"
    "                print(f'LLaVA error: {e}')\n"
    "                captions = ['']\n"
    "        else:\n"
    "            captions = ['']"
)
if old_captions_block in content:
    content = content.replace(old_captions_block, new_captions_block)
else:
    old_captions_block_alt = (
        "    if use_llava:\n"
        "        captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
        "    else:\n"
        "        captions = ['']"
    )
    new_captions_block_alt = (
        "    if use_llava and llava_agent is not None:\n"
        "        try:\n"
        "            captions = llava_agent.gen_image_caption([clean_PIL_img])\n"
        "            print(f'>>> LLaVA CAPTION: {captions}')\n"
        "            if hasattr(llava_agent, \"model\"):\n"
        "                del llava_agent.model\n"
        "            del llava_agent\n"
        "            llava_agent = None\n"
        "            import torch, gc\n"
        "            gc.collect()\n"
        "            torch.cuda.empty_cache()\n"
        "        except Exception as e:\n"
        "            print(f'LLaVA error: {e}')\n"
        "            captions = ['']\n"
        "    else:\n"
        "        captions = ['']"
    )
    if old_captions_block_alt in content:
        content = content.replace(old_captions_block_alt, new_captions_block_alt)
with open("test.py", "w") as f:
    f.write(content)

# Patch 5: SUPIR/utils/tilevae.py
tilevae_path = os.path.join("SUPIR", "utils", "tilevae.py")
with open(tilevae_path, "r") as f:
    tilevae_content = f.read()
tilevae_content = tilevae_content.replace("\r\n", "\n")
old_block = '''        if is_xformers_available:
            # task_queue.append(('attn', lambda x, net=net: attn_forward_new_xformers(net, x)))
            task_queue.append(
                ('attn', lambda x, net=net: xformer_attn_forward(net, x)))
        elif hasattr(F, "scaled_dot_product_attention"):
            task_queue.append(('attn', lambda x, net=net: attn_forward_new_pt2_0(net, x)))
        else:
            task_queue.append(('attn', lambda x, net=net: attn_forward_new(net, x)))'''
new_block = '''        task_queue.append(('attn', lambda x, net=net: attn_forward(net, x)))'''
if old_block in tilevae_content:
    tilevae_content = tilevae_content.replace(old_block, new_block)
with open(tilevae_path, "w") as f:
    f.write(tilevae_content)

# Patch 5b: sgm/modules/diffusionmodules/model.py
diff_model_path = os.path.join("sgm", "modules", "diffusionmodules", "model.py")
with open(diff_model_path, "r") as f:
    diff_model_content = f.read()
diff_model_content = diff_model_content.replace("\r\n", "\n")
old_func_def = 'def make_attn(in_channels, attn_type="vanilla", attn_kwargs=None):'
new_func_def = (
    'def make_attn(in_channels, attn_type="vanilla", attn_kwargs=None):\n'
    '    if attn_type == "vanilla-xformers" and not XFORMERS_IS_AVAILABLE:\n'
    '        attn_type = "vanilla"'
)
if old_func_def in diff_model_content:
    diff_model_content = diff_model_content.replace(old_func_def, new_func_def)
else:
    old_func_def_single = "def make_attn(in_channels, attn_type='vanilla', attn_kwargs=None):"
    if old_func_def_single in diff_model_content:
        diff_model_content = diff_model_content.replace(old_func_def_single, new_func_def)
with open(diff_model_path, "w") as f:
    f.write(diff_model_content)

# Patch 6: sgm/modules/encoders/modules.py
modules_path = os.path.join("sgm", "modules", "encoders", "modules.py")
with open(modules_path, "r") as f:
    modules_content = f.read()
modules_content = modules_content.replace("\r\n", "\n")
old_call = '''        model, _, _ = open_clip.create_model_and_transforms(
            arch,
            device=torch.device("cpu"),
            pretrained=version if SDXL_CLIP2_CKPT_PTH is None else SDXL_CLIP2_CKPT_PTH,
        )
        del model.visual'''
new_call = '''        model, _, _ = open_clip.create_model_and_transforms(
            arch,
            device=torch.device("cuda"),
            pretrained="" if SDXL_CLIP2_CKPT_PTH is None else SDXL_CLIP2_CKPT_PTH,
            precision="fp16",
        )
        model = model.half()
        del model.visual'''
if old_call in modules_content:
    modules_content = modules_content.replace(old_call, new_call)
modules_content = modules_content.replace("model = model.float()", "model = model.half()")
with open(modules_path, "w") as f:
    f.write(modules_content)

# Patch 7: SUPIR/util.py
util_path = os.path.join("SUPIR", "util.py")
with open(util_path, "r") as f:
    util_content = f.read()
util_content = util_content.replace("\r\n", "\n")
old_load_state_dict = """def load_state_dict(ckpt_path, location='cpu'):
    if ckpt_path.endswith('.safetensors'):
        state_dict = load_safetensors(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    return state_dict"""
new_load_state_dict = """def load_state_dict(ckpt_path, location='cpu'):
    if ckpt_path.endswith('.safetensors'):
        state_dict = load_safetensors(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    for k in list(state_dict.keys()):
        if hasattr(state_dict[k], 'half'):
            state_dict[k] = state_dict[k].half()
    return state_dict"""
if old_load_state_dict in util_content:
    util_content = util_content.replace(old_load_state_dict, new_load_state_dict)
else:
    old_load_state_dict_double = """def load_state_dict(ckpt_path, location="cpu"):
    if ckpt_path.endswith('.safetensors'):
        state_dict = load_safetensors(ckpt_path, device=location)
    else:
        state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))
    return state_dict"""
    if old_load_state_dict_double in util_content:
        util_content = util_content.replace(old_load_state_dict_double, new_load_state_dict)

util_content = util_content.replace(
    "model = instantiate_from_config(config.model).cpu()",
    "import torch\n    old_dtype = torch.get_default_dtype()\n    torch.set_default_dtype(torch.float16)\n    try:\n        with torch.device('cuda'):\n            model = instantiate_from_config(config.model)\n    finally:\n        torch.set_default_dtype(old_dtype)"
)
util_content = util_content.replace(
    "model.load_state_dict(load_state_dict(config.SDXL_CKPT), strict=False)",
    "model.load_state_dict(load_state_dict(config.SDXL_CKPT), strict=False)\n        import gc, torch\n        gc.collect()\n        torch.cuda.empty_cache()"
)
util_content = util_content.replace(
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_F), strict=False)",
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_F), strict=False)\n            import gc, torch\n            gc.collect()\n            torch.cuda.empty_cache()"
)
util_content = util_content.replace(
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_Q), strict=False)",
    "model.load_state_dict(load_state_dict(config.SUPIR_CKPT_Q), strict=False)\n            import gc, torch\n            gc.collect()\n            torch.cuda.empty_cache()"
)
with open(util_path, "w") as f:
    f.write(util_content)

# Patch 8: sgm/modules/diffusionmodules/util.py
util_diff_path = os.path.join("sgm", "modules", "diffusionmodules", "util.py")
with open(util_diff_path, "r") as f:
    content = f.read()
content = content.replace("\r\n", "\n")
content = content.replace(
    "return betas.numpy()",
    "return betas.cpu().numpy()"
)
with open(util_diff_path, "w") as f:
    f.write(content)

print("All patches applied. Installation complete.")

In [ ]:
#@title ##**Select Video File** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

upload_option = "Load from Google Drive Root"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive"]

file_name = None
last_selected_button = None

def reset_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

if upload_option == "Upload from PC":
    print("Please upload a video file.")
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
    else:
        print("No file uploaded.")
        file_name = None

elif upload_option == "Load from Google Drive Root":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for f in os.listdir(root_dir):
        if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in video_extensions:
            files_list.append(f)

    if not files_list:
        print("No video files found in Google Drive root.")
        file_name = None
    else:
        print("Select a video file from Google Drive root:")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

elif upload_option == "Load from Google Drive":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in video_extensions:
                relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                files_list.append(relative_path)

    if not files_list:
        print("No video files found in Google Drive or its subfolders.")
        file_name = None
    else:
        print("Select a video file from Google Drive (including subfolders):")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if file_name:
    print(f"Video file path set to: {file_name}")
else:
    print("Video file path not set. Please select a file.")

In [ ]:
#@title ##**Config Folders** { display-mode: "form" }
import os
import shutil
from google.colab import drive

output_folder = "google_drive" #@param ["google_drive","root"]

if output_folder == "google_drive":
    if not os.path.exists('/content/drive'):
        print("Google Drive не подключён. Подключаем...")
        drive.mount('/content/drive')
    real_output_folder = '/content/drive/MyDrive/supir_output'
    real_input_folder = "/content/drive/MyDrive/supir_input"
elif output_folder == "root":
    real_output_folder = '/content/supir_output'
    real_input_folder = "/content/supir_input"

if not os.path.exists(real_output_folder):
    os.makedirs(real_output_folder)

if not os.path.exists(real_input_folder):
    os.makedirs(real_input_folder)

#clear folders
clear_input_folder = False #@param {type:"boolean"}
up_to_frame = "" #@param {type:"string"}
from_frame = "" #@param {type:"string"}

def clean_folder(folder_path, up_to=None, from_frame=None):
    print(f"\nCurrent parameters:")
    print(f"Delete frames up to: {up_to if up_to else 'not specified'}")
    print(f"Delete frames after: {from_frame if from_frame else 'not specified'}")

    if not os.path.isdir(folder_path):
        print(f"\nFolder {folder_path} does not exist!")
        print("Creating a new folder...")
        os.makedirs(folder_path)
        return

    if not up_to and not from_frame:
        print("\nNo parameters specified - deleting all folder content...")
        shutil.rmtree(folder_path)
        os.makedirs(folder_path)
        print(f"Folder {folder_path} cleared and recreated")
        return

    print("\nStarting file processing...")
    files = os.listdir(folder_path)
    jpg_files = [f for f in files if f.endswith('.jpg')]

    if not jpg_files:
        print("No JPG files to process in the folder")
        return

    deleted_count = 0
    processed_count = 0

    for filename in jpg_files:
        try:
            frame_number = int(filename.split('.')[0])
            should_delete = False

            if up_to and from_frame:
                if frame_number < int(up_to) or frame_number > int(from_frame):
                    should_delete = True
            elif up_to:
                if frame_number < int(up_to):
                    should_delete = True
            elif from_frame:
                if frame_number > int(from_frame):
                    should_delete = True

            if should_delete:
                file_path = os.path.join(folder_path, filename)
                os.remove(file_path)
                deleted_count += 1
                if deleted_count <= 5:
                    print(f'File deleted: {filename}')
                elif deleted_count == 6:
                    print('...')
            else:
                processed_count += 1

        except ValueError:
            print(f'Skipped file with invalid name: {filename}')

    print(f'\nProcessing complete:')
    print(f'Total files: {len(jpg_files)}')
    print(f'Files deleted: {deleted_count}')
    print(f'Files retained: {processed_count}')

if clear_input_folder:
    up_to_frame = up_to_frame if up_to_frame != "0" else None
    from_frame = from_frame if from_frame != "0" else None
    clean_folder(real_input_folder, up_to_frame, from_frame)

clear_output_folder = False #@param {type:"boolean"}

if clear_output_folder:
    if os.path.isdir(real_output_folder):
        shutil.rmtree(real_output_folder)
    os.makedirs(real_output_folder)

In [ ]:
#@title ##**Run sequence (Extract Frames)** { display-mode: "form" }
import cv2
import imageio
import os
import tqdm
import subprocess
import numpy as np
import time

library = "ffmpeg" #@param ["cv2","imageio","ffmpeg","skvideo","scipy","moviepy"]
delay = "0.1" #@param [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

if (library == "ffmpeg"):
    import ffmpeg
    # Using file_name set from Cell 2
    full_path = file_name

    probe = ffmpeg.probe(full_path)
    video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    fps = video_info['r_frame_rate']
    duration = float(video_info['duration'])
    frame_count = int(video_info['nb_frames'])

    print("FPS: ", fps)
    print("Duration: ", duration)
    print("Frames: ", frame_count)

    pbar_ffmpeg = tqdm.tqdm(total=frame_count, ncols=100, position=0, leave=True)
    process = (
        ffmpeg
        .input(full_path)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24', qscale=0)
        .run_async(pipe_stdout=True)
    )

    for i in range(frame_count):
        try:
            raw_video = process.stdout.read(video_info['width'] * video_info['height'] * 3)
            frame = np.frombuffer(raw_video, dtype='uint8').reshape((video_info['height'], video_info['width'], 3))
            frame_path = f"{real_input_folder}/{i:09d}.jpg"
            if os.path.isfile(frame_path):
              pbar_ffmpeg.update(1)
              continue
            imageio.imwrite(frame_path, frame)
        except Exception as e:
            print(f"Error writing to disk: {str(e)}. Retrying...")
            continue
        pbar_ffmpeg.update(1)
        time.sleep(float(delay))

    pbar_ffmpeg.close()
    process.wait()

#check frames
def check_frames():
    frame_dir = real_input_folder
    frames = [int(f.split('.')[0].replace('frame', '')) for f in os.listdir(frame_dir) if f.lower().endswith(('.jpg', '.png'))]
    if not frames:
        print("No frames found.")
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print(min_frame)
    print(max_frame)

    missing_frames = []
    for i in range(min_frame, max_frame+1):
        if i not in frames:
            missing_frames.append(i)

    if len(missing_frames) > 0:
        print(f"Missing frames: {missing_frames}")
    else:
        print("All frames present")

attempts = 0
max_attempts = 10

while attempts < max_attempts:
    try:
        check_frames()
        break
    except Exception as e:
        attempts += 1
        print(f"Attempt {attempts} failed with error: {str(e)}")
        if attempts == max_attempts:
            print("Maximum attempts reached. Execution failed.")
        else:
            print("Retrying...")

In [ ]:
#@title ##**Run SUPIR Upscale on Frames** { display-mode: "form" }
import os
import shutil
import subprocess
import glob
import re
import collections
import time
import torch
import gc
from tqdm import tqdm

# SUPIR Parameters
SUPIR_sign = "F" #@param ["Q", "F"] {type:"string"}
upscale = 3 #@param {type:"slider", min:1, max:8, step:0.5}
min_size = 1024 #@param {type:"integer"}
edm_steps = 100 #@param {type:"slider", min:10, max:100, step:1}

# Sharpness and fidelity settings
s_cfg = 4 #@param {type:"slider", min:1.0, max:15.0, step:0.1}
s_stage2 = 0.95 #@param {type:"slider", min:0.0, max:1.0, step:0.01}
s_stage1 = -1 #@param {type:"integer"}

# Memory optimization (Tiling)
decoder_tile_size = 64 #@param [64, 96, 128, 192, 256] {type:"raw"}
encoder_tile_size = 512 #@param [256, 512, 1024] {type:"raw"}

# Text prompts (Universal)
no_llava = True #@param {type:"boolean"}
my_positive_prompt = "hyperrealism" #@param {type:"string"}
my_negative_prompt = "blur" #@param {type:"string"}

batch_size = 1 #@param {type:"slider", min:1, max:10, step:1}
show_full_log = False #@param {type:"boolean"}

# Универсальное извлечение номера кадра из любого имени
def extract_frame_number(filename):
    match = re.search(r'\d+', filename)
    return int(match.group()) if match else None

# Check frames output directory
def check_frames_output():
    frame_dir = real_input_folder
    frames = []
    for f in os.listdir(frame_dir):
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            num = extract_frame_number(f)
            if num is not None:
                frames.append(num)

    if not frames:
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print("Input range:", min_frame, "-", max_frame)

    missing_frames = []
    for i in range(min_frame, max_frame+1):
        if i not in frames:
            missing_frames.append(i)

    if len(missing_frames) > 0:
        print(f"Missing frames: {missing_frames}")
    else:
        print("All frames present")

check_frames_output()

# Folders for SUPIR script batching
upload_folder = "/content/SUPIR/input_images"
result_folder = "/content/SUPIR/output_images"

file_list = os.listdir(real_input_folder)
file_list.sort()
frames = []
for f in file_list:
    if f.lower().endswith(('.jpg', '.png', '.jpeg')):
        num = extract_frame_number(f)
        if num is not None:
            frames.append(num)
min_frame = min(frames) if frames else 0

real_files = os.listdir(real_output_folder)
if real_files:
    real_frames = []
    for f in real_files:
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            num = extract_frame_number(f)
            if num is not None:
                real_frames.append(num)
    start_frame = max(real_frames) + 1 if real_frames else min_frame
else:
    start_frame = min_frame

max_frame = frames[-1] if frames else 0
files_to_copy = [f"{real_input_folder}/{frame:09d}.jpg" for frame in range(start_frame, max_frame+1) if f"{frame:09d}.jpg" in file_list]

total_files = len(files_to_copy)
print(f"Resuming from frame {start_frame}. Total frames left: {total_files}")

%cd /content/SUPIR

i = 0
retry_count = 0
max_retries = 3

# Оптимизация памяти PyTorch от фрагментации
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

with tqdm(total=total_files, desc="Processing frames") as pbar:
    while i < total_files:
        batch_files = files_to_copy[i:i+batch_size]

        if os.path.isdir(upload_folder):
            shutil.rmtree(upload_folder)
        os.makedirs(upload_folder)
        if os.path.isdir(result_folder):
            shutil.rmtree(result_folder)
        os.makedirs(result_folder)

        for file in batch_files:
            shutil.copy(file, upload_folder)

        # Очистка памяти перед запуском
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        cmd = [
            "python", "-u", "test.py",
            "--img_dir", upload_folder,
            "--save_dir", result_folder,
            "--SUPIR_sign", str(SUPIR_sign),
            "--upscale", str(upscale),
            "--min_size", str(min_size),
            "--edm_steps", str(edm_steps),
            "--s_cfg", str(s_cfg),
            "--s_stage2", str(s_stage2),
            "--loading_half_params",
            "--use_tile_vae",
            "--encoder_tile_size", str(encoder_tile_size),
            "--decoder_tile_size", str(decoder_tile_size),
            "--a_prompt", str(my_positive_prompt),
            "--n_prompt", str(my_negative_prompt),
        ]

        if s_stage1 > 0:
            cmd.extend(["--s_stage1", str(s_stage1)])
        if no_llava:
            cmd.append("--no_llava")

        process = None
        log_queue = collections.deque(maxlen=30)

        try:
            # Запуск скрипта
            process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

            for line in process.stdout:
                if show_full_log:
                    print(line, end="")
                log_queue.append(line)

            process.wait()

        except KeyboardInterrupt:
            print("\n[ОСТАНОВКА] Прервано пользователем. Очистка...")
            raise

        finally:
            if process is not None and process.poll() is None:
                process.terminate()
                process.wait()

        # Проверка результатов
        output_files = os.listdir(result_folder) if os.path.exists(result_folder) else []

        if process.returncode != 0 or len(output_files) != len(batch_files):
            retry_count += 1
            print(f"\n[ОШИБКА] Обработка прервалась. Попытка {retry_count} из {max_retries}.")

            if not show_full_log:
                print("--- ПОСЛЕДНИЕ СТРОКИ ЛОГА ---")
                for l in log_queue:
                    print(l, end="")
                print("-----------------------------")

            if retry_count >= max_retries:
                raise RuntimeError("Критическая ошибка (OOM). Уменьшите upscale или encoder_tile_size.")

            time.sleep(2)
            continue # Повтор того же кадра

        # Если успех - записываем файлы на диск СРАЗУ с правильными чистыми именами
        retry_count = 0
        for out_file in output_files:
            if out_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                frame_num = extract_frame_number(out_file)
                if frame_num is not None:
                    ext = out_file.split('.')[-1]
                    clean_name = f"{frame_num:09d}.{ext}"
                    # Копируем файл из временной папки на Гугл Диск с новым чистым именем
                    shutil.copy(
                        os.path.join(result_folder, out_file),
                        os.path.join(real_output_folder, clean_name)
                    )

        i += len(batch_files)
        pbar.update(len(batch_files))

# Функция финальной проверки кадров в папке вывода
def check_final_frames():
    frame_dir = real_output_folder
    frames = []
    for f in os.listdir(frame_dir):
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            num = extract_frame_number(f)
            if num is not None:
                frames.append(num)

    if not frames:
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print("\nOutput range:", min_frame, "-", max_frame)

    missing_frames = []
    for i in range(min_frame, max_frame+1):
        if i not in frames:
            missing_frames.append(i)

    if len(missing_frames) > 0:
        print(f"Missing output frames: {missing_frames}")
    else:
        print("All output frames present")

check_final_frames()

In [ ]:
#@title ##**Create video** { display-mode: "form" }
import cv2
import os
import subprocess
import time
from tqdm.notebook import tqdm
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

def log_time(start, message):
    elapsed = time.time() - start
    print(f"{message}: {elapsed:.2f} seconds")
    return time.time()

start_time = time.time()

upscaled_image = 100 #@param {type:"slider", min:0, max:100, step:1}

print(f"output_folder: {output_folder}")

if 'file_name' in locals() and os.path.exists(file_name):
    base_file_name = os.path.basename(file_name)
else:
    raise ValueError("file_name is not defined or the file does not exist")

if output_folder == "google_drive":
    save_path = '/content/drive/MyDrive/'
elif output_folder == "root":
    save_path = '/content/'
else:
    save_path = '/content/'

full_path = os.path.join(save_path, base_file_name) if not os.path.exists(file_name) else file_name
output_file_name = base_file_name.rsplit('.', 1)[0] + f'_SUPIR_upscale_{upscaled_image}.mp4'
output_file = os.path.join(save_path, output_file_name)
temp_video = "/content/temp_video.mp4"

start_time = log_time(start_time, "Initial setup")

cap = cv2.VideoCapture(full_path)
fps_of_video = int(cap.get(cv2.CAP_PROP_FPS))
cap.release()

upscaled_img_files = [os.path.join(real_output_folder, img) for img in os.listdir(real_output_folder) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
upscaled_img_files.sort()

if upscaled_image < 100:
    original_img_files = [os.path.join(real_input_folder, img) for img in os.listdir(real_input_folder) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
    original_img_files.sort()
    if len(upscaled_img_files) != len(original_img_files):
        raise ValueError("Number of upscaled and original frames does not match")

if upscaled_img_files:
    first_frame = cv2.imread(upscaled_img_files[0], cv2.IMREAD_COLOR)
    height, width = first_frame.shape[:2]

    needs_resize = False
    for img in upscaled_img_files[:10]:
        frame = cv2.imread(img, cv2.IMREAD_COLOR)
        if frame.shape[:2] != (height, width):
            needs_resize = True
            break
        del frame
    del first_frame
else:
    raise ValueError("No images found in the upscaled frames folder")

start_time = log_time(start_time, "Frame list preparation")

def get_video_bitrate(file_path):
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', file_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    bitrate = result.stdout.strip()
    try:
        return int(bitrate)
    except ValueError:
        return None

bitrate = get_video_bitrate(full_path)
if bitrate:
    bitrate = int(bitrate * 1.5)
    bitrate_str = f'{bitrate // 1000}k'
else:
    bitrate_str = '7500k'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_video, fourcc, fps_of_video, (width, height))

if upscaled_image == 100:
    for img_file in tqdm(upscaled_img_files, desc="Processing frames"):
        frame = cv2.imread(img_file, cv2.IMREAD_COLOR)
        if needs_resize and frame.shape[:2] != (height, width):
            frame = cv2.resize(frame, (width, height))
        out.write(frame)
        del frame
else:
    alpha = upscaled_image / 100.0
    beta = 1 - alpha
    for upscaled_img, original_img in tqdm(zip(upscaled_img_files, original_img_files), total=len(upscaled_img_files), desc="Processing frames"):
        upscaled_frame = cv2.imread(upscaled_img, cv2.IMREAD_COLOR)
        original_frame = cv2.imread(original_img, cv2.IMREAD_COLOR)

        if needs_resize and upscaled_frame.shape[:2] != (height, width):
            upscaled_frame = cv2.resize(upscaled_frame, (width, height))

        original_frame_resized = cv2.resize(original_frame, (width, height))

        blended_frame = cv2.addWeighted(upscaled_frame, alpha, original_frame_resized, beta, 0)

        out.write(blended_frame)
        del upscaled_frame, original_frame, original_frame_resized, blended_frame

out.release()
gc.collect()

start_time = log_time(start_time, "Frame processing and writing")

temp_converted = "/content/temp_converted.mp4"
cmd = ['ffmpeg', '-i', temp_video, '-c:v', 'libx264', '-b:v', bitrate_str, '-y', temp_converted]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg conversion failed: {result.stderr}")
    raise RuntimeError("Conversion to libx264 failed")
os.remove(temp_video)
os.rename(temp_converted, temp_video)

start_time = log_time(start_time, "FFmpeg conversion to libx264")

cmd = ['ffmpeg', '-i', temp_video, '-i', full_path, '-map', '0:v', '-map', '1:a?', '-map', '1:s?', '-c', 'copy', '-y', output_file]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg audio muxing failed: {result.stderr}")
    raise RuntimeError("Audio muxing failed")

start_time = log_time(start_time, "Final audio and subtitles muxing")

if os.path.exists(output_file):
    if os.path.exists(temp_video):
        os.remove(temp_video)
    print("Video created successfully")
    print(f"Video saved at: {output_file}")
else:
    print("Failed to create video")
    print(f"Expected save path: {output_file}")
    print(f"FFmpeg error output: {result.stderr}")

start_time = log_time(start_time, "Cleanup")